# Лабораторна робота №5: Мультимодальні моделі для генерації описів зображень (Image Captioning)
## Частина 2: Контрольовані експерименти з формулюванням промпту, `temperature` та `max_new_tokens`

**Дисципліна:** Нейромережеві технології і системи  
**Тема:** Мультимодальні архітектури та генерація описів зображень  
**Варіант:** №2 — Генерація описів для кількох зображень з дослідженням впливу формулювання промпту, `temperature` та `max_new_tokens`  
**Студент:** Max  
**Університет:** Національний університет «Львівська політехніка»  

---

### Мета другого етапу:
1. Провести систематичне дослідження впливу трьох стилів формулювання текстових промптів (**concise**, **descriptive**, **detailed**) на повноту та лексичне багатство описів усіх 4 зображень.
2. Дослідити вплив температури стохастичного декодування $T \in [0.0, 0.2, 0.7, 1.0]$ на лексичну різноманітність та ризик виникнення візуальних галюцинацій при фіксованому `seed=42`.
3. Вивчити залежність довжини генерованого тексту, часу інференсу та якості завершення від виділеного бюджету вихідних токенів `max_new_tokens` $\in [32, 64, 128, 256]$.
4. Зафіксувати єдиний структурований CSV- та JSON-реєстр усіх експериментів із фіксацією параметрів, часу виконання та кількості токенів.

## 1. Завантаження моделі та налаштування експериментальної матриці

In [1]:
import time
import json
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from transformers import AutoProcessor, LlavaForConditionalGeneration

def resolve_lab_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / 'inputs' / '01_coastal_lighthouse.png').exists() and candidate.name == 'lab5_multimodal_captioning':
            return candidate
        if (candidate / 'lab5_multimodal_captioning' / 'inputs' / '01_coastal_lighthouse.png').exists():
            return candidate / 'lab5_multimodal_captioning'
    return cwd

LAB_DIR = resolve_lab_dir()
INPUTS_DIR = (LAB_DIR / "inputs").resolve()
OUTPUTS_DIR = (LAB_DIR / "outputs").resolve()
TABLES_DIR = (OUTPUTS_DIR / "tables").resolve()
METADATA_DIR = (OUTPUTS_DIR / "metadata").resolve()

csv_path = TABLES_DIR / "caption_experiments.csv"
assert csv_path.exists(), f"Файл експериментів не знайдено: {csv_path}"

exp_df = pd.read_csv(csv_path)
print(f"Базовий каталог ЛР5: {LAB_DIR}")
print(f"Успішно завантажено реєстр з {len(exp_df)} експериментальних запусків.")
exp_df.head(4)

C:\Users\Eclipse\PycharmProjects\neural-network-technologies-and-systems\.venv-lab04\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Базовий каталог ЛР5: C:\Users\Eclipse\PycharmProjects\neural-network-technologies-and-systems\max\lab5_multimodal_captioning
Успішно завантажено реєстр з 20 експериментальних запусків.


,run_id,image_id,image_file,category,series,prompt_style,prompt_text,temperature,max_new_tokens,generated_tokens,char_count,latency_sec,tokens_per_sec,caption,model_id,device
0,img1_lighthouse_prompt_concise,img1_lighthouse,01_coastal_lighthouse.png,Landscape / Maritime,prompt_wording,concise,"Provide a concise, single-sentence caption des...",0.0,128,11,46,2.1464,5.12,A lighthouse on a cliff overlooking the ocean.,llava-hf/llava-interleave-qwen-0.5b-hf,cuda:0
1,img1_lighthouse_prompt_descriptive,img1_lighthouse,01_coastal_lighthouse.png,Landscape / Maritime,prompt_wording,descriptive,Describe this image in a well-structured parag...,0.0,128,128,612,7.5330,16.99,The image presents a striking scene of a light...,llava-hf/llava-interleave-qwen-0.5b-hf,cuda:0
2,img1_lighthouse_prompt_detailed,img1_lighthouse,01_coastal_lighthouse.png,Landscape / Maritime,prompt_wording,detailed,"Provide an exhaustive, highly detailed descrip...",0.0,128,128,596,6.9473,18.42,The image presents a striking scene of a light...,llava-hf/llava-interleave-qwen-0.5b-hf,cuda:0
3,img2_cafe_prompt_concise,img2_cafe,02_cafe_bicycle.png,Urban / Street Scene,prompt_wording,concise,"Provide a concise, single-sentence caption des...",0.0,128,12,46,0.8316,14.43,A bicycle parked next to a table and a window.,llava-hf/llava-interleave-qwen-0.5b-hf,cuda:0


## 2. Серія А: Дослідження впливу формулювання промпту (Prompt Wording)

Порівнюються 3 стилістичні парадигми на всіх 4 зображеннях (загалом 12 запусків при $T=0.0$, `max_new_tokens=128`):
1. **Concise:** *"Provide a concise, single-sentence caption describing this image."*
2. **Descriptive:** *"Describe this image in a well-structured paragraph, highlighting key objects, colors, and setting."*
3. **Detailed:** *"Provide an exhaustive, highly detailed description of all visible objects, spatial relationships, materials, and lighting in this image."*

In [2]:
series_a = exp_df[exp_df["series"] == "prompt_wording"]
print("Зведена таблиця серії А (вплив формулювання промпту):")
display_cols = ["image_id", "prompt_style", "generated_tokens", "char_count", "latency_sec", "tokens_per_sec"]
series_a[display_cols]

Зведена таблиця серії А (вплив формулювання промпту):


,image_id,prompt_style,generated_tokens,char_count,latency_sec,tokens_per_sec
0,img1_lighthouse,concise,11,46,2.1464,5.12
1,img1_lighthouse,descriptive,128,612,7.5330,16.99
2,img1_lighthouse,detailed,128,596,6.9473,18.42
3,img2_cafe,concise,12,46,0.8316,14.43
4,img2_cafe,descriptive,128,613,7.2121,17.75
5,img2_cafe,detailed,128,575,6.7524,18.96
6,img3_library,concise,11,38,0.7645,14.39
7,img3_library,descriptive,119,577,6.6789,17.82
8,img3_library,detailed,128,609,7.0063,18.27
9,img4_robotics,concise,12,44,0.9253,12.97


In [3]:
for _, row in series_a[series_a["image_id"] == "img1_lighthouse"].iterrows():
    print(f"=== Стиль: {row['prompt_style'].upper()} (Токенів: {row['generated_tokens']}, Час: {row['latency_sec']} с) ===")
    print(f"{row['caption']}\n")

=== Стиль: CONCISE (Токенів: 11, Час: 2.1464 с) ===
A lighthouse on a cliff overlooking the ocean.

=== Стиль: DESCRIPTIVE (Токенів: 128, Час: 7.533 с) ===
The image presents a striking scene of a lighthouse against a backdrop of a sunset. The lighthouse, a classic symbol of maritime safety, stands tall on a rocky outcrop. Its pointed roof and two red stripes add a touch of color to the otherwise monochromatic scene. The lighthouse is facing towards the right side of the image, as if ready to guide ships at sea. The sunset, painted in hues of orange and yellow, casts a warm glow over the scene, enhancing the overall tranquility of the image. The rocky outcrop on which the lighthouse stands adds a rugged texture to the composition, contrasting with the smooth

=== Стиль: DETAILED (Токенів: 128, Час: 6.9473 с) ===
The image presents a striking scene of a lighthouse against the backdrop of a sunset. The lighthouse, a classic symbol of maritime safety, stands tall on a rocky outcrop. Its p

## 3. Серія B: Дослідження впливу температури стохастичного декодування (`temperature`)

Для оцінки впливу стохастичного семплінгу проведено розгортку значень $T \in [0.0, 0.2, 0.7, 1.0]$ на зображенні Маяка з дескриптивним промптом при фіксованому генераторі випадкових чисел:
- **$T = 0.0$ (Greedy search):** повністю детермінований вибір токена з максимальною ймовірністю $P(w_t)$.
- **$T = 0.2$:** незначне згладжування розподілу Softmax, легка стилістична варіативність при збереженні максимальної фактологічної консервативності.
- **$T = 0.7$:** помітне збагачення словникового запасу, поява образних метафор («stylized representation», «serene backdrop»).
- **$T = 1.0$:** висока стохастичність, збільшення ентропії тексту та підвищений ризик відхилення від прямих візуальних фактів.

In [4]:
series_b = exp_df[exp_df["series"] == "temperature"].sort_values("temperature")
print("Зведена таблиця серії B (вплив температури):")
series_b[["run_id", "temperature", "generated_tokens", "char_count", "latency_sec", "tokens_per_sec"]]

Зведена таблиця серії B (вплив температури):


,run_id,temperature,generated_tokens,char_count,latency_sec,tokens_per_sec
12,img1_lighthouse_temp_0_0,0.0,128,612,7.1654,17.86
13,img1_lighthouse_temp_0_2,0.2,128,625,7.0994,18.03
14,img1_lighthouse_temp_0_7,0.7,114,532,6.3740,17.89
15,img1_lighthouse_temp_1_0,1.0,128,598,7.6287,16.78


In [5]:
for _, row in series_b.iterrows():
    print(f"=== Temperature = {row['temperature']} (Символів: {row['char_count']}, Час: {row['latency_sec']} с) ===")
    print(f"{row['caption']}\n")

=== Temperature = 0.0 (Символів: 612, Час: 7.1654 с) ===
The image presents a striking scene of a lighthouse against a backdrop of a sunset. The lighthouse, a classic symbol of maritime safety, stands tall on a rocky outcrop. Its pointed roof and two red stripes add a touch of color to the otherwise monochromatic scene. The lighthouse is facing towards the right side of the image, as if ready to guide ships at sea. The sunset, painted in hues of orange and yellow, casts a warm glow over the scene, enhancing the overall tranquility of the image. The rocky outcrop on which the lighthouse stands adds a rugged texture to the composition, contrasting with the smooth

=== Temperature = 0.2 (Символів: 625, Час: 7.0994 с) ===
The image presents a striking scene of a lighthouse against the backdrop of a sunset. The lighthouse, a classic symbol of maritime navigation, stands tall on a rocky outcrop. Its pointed roof and red and white striped sails add a touch of color to the otherwise monochroma

## 4. Серія C: Дослідження впливу бюджету токенів (`max_new_tokens`)

Досліджено поведінку моделі при зміні граничної кількості генерованих токенів `max_new_tokens` $\in [32, 64, 128, 256]$ на складному інженерному зображенні лабораторії робототехніки:
- **32 токени:** передчасне обривання речення (штучне усічення думки моделлю).
- **64 токени:** частковий абзац, незавершений фінальний висновок.
- **128 токенів:** збалансований структурований опис ключових об'єктів.
- **256 токенів:** модель повністю описує сцену та природно завершує генерацію генератором токена кінця послідовності `<|im_end|>` (фактично 208 токенів), не заповнюючи весь ліміт штучним текстом.

In [6]:
series_c = exp_df[exp_df["series"] == "token_budget"].sort_values("max_new_tokens")
print("Зведена таблиця серії C (бюджет токенів):")
series_c[["run_id", "max_new_tokens", "generated_tokens", "latency_sec", "tokens_per_sec", "char_count"]]

Зведена таблиця серії C (бюджет токенів):


,run_id,max_new_tokens,generated_tokens,latency_sec,tokens_per_sec,char_count
16,img4_robotics_tokens_32,32,32,1.9737,16.21,137
17,img4_robotics_tokens_64,64,64,3.6193,17.68,268
18,img4_robotics_tokens_128,128,128,7.6903,16.64,582
19,img4_robotics_tokens_256,256,208,11.0402,18.84,999


In [7]:
for _, row in series_c.iterrows():
    print(f"=== Бюджет: {row['max_new_tokens']} токенів (Фактично: {row['generated_tokens']}, Час: {row['latency_sec']} с) ===")
    print(f"{row['caption']}\n")

=== Бюджет: 32 токенів (Фактично: 32, Час: 1.9737 с) ===
The image presents a scene set in a dimly lit room, with a focus on a robotic arm and a digital display. The robotic arm, which is orange

=== Бюджет: 64 токенів (Фактично: 64, Час: 3.6193 с) ===
The image presents a scene set in a dimly lit room, with a focus on a robotic arm and a digital display. The robotic arm, which is orange, is positioned to the left of the frame, and it appears to be in motion, as indicated by the green and red lights on its arm. 

To

=== Бюджет: 128 токенів (Фактично: 128, Час: 7.6903 с) ===
The image presents a scene set in a dimly lit room, with a focus on a robotic arm and a digital display. The robotic arm, which is orange, is positioned to the left of the frame, and it appears to be in motion, as indicated by the green and red lights on its arm. 

To the right of the robotic arm, there is a digital display. The display is green and white, and it shows a waveform graph, which is a graphical represe